# CALE Behavior Matrix Results

This notebook organizes the current neutral full FEVER behavior-matrix analysis into a paper-facing experiment story.

本 notebook 把当前 neutral full FEVER 的 behavior-matrix analysis 整理成可以写进论文的实验故事。

The core question is not only which evaluator has the highest score, but whether evaluator behavior can be decomposed, diagnosed, and modeled as measurement evidence.

核心问题不只是哪个 evaluator 分数最高，而是 evaluator behavior 是否可以被拆解、诊断，并作为 measurement evidence 来建模。

## Experiment Roadmap

**Experiment 0: Data and Output Audit.** We verify the response file, evaluator variants, target models, and behavior matrix size.

**Experiment 1: Shared Outcome-Level Comparison.** We compare outputs available for all evaluator variants, such as final score and quality tier, while avoiding factual-status interpretations of quality tiers.

**Experiment 2: Diagnostic Availability and Behavior Profiles.** We show which evaluator variants produce construct-level observables, then compare CALE behavior profiles.

**Experiment 3: Latent Organization of CALE Behavior.** We run correlation and PCA on CALE behavior variables after excluding reference-label indicator columns.

**Experiment 4: Writeup Tables.** We save compact tables and figures for the paper draft.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

RESPONSE_PATH = Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full.jsonl')
REPORT_PATH = Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_report.json')
BEHAVIOR_PATH = Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv')
OUTDIR = Path('figures/behavior_matrix_results')
OUTDIR.mkdir(parents=True, exist_ok=True)

CORE_BEHAVIOR_COLUMNS = [
    'misinformation_detection',
    'framing_resistance',
    'claim_status_recognition',
    'error_rejection',
    'correction_accuracy',
    'evidence_grounding',
    'source_faithfulness',
    'hallucination_control',
    'uncertainty_handling',
]

PROXY_COLUMNS = [
    'nei_uncertainty_failure_proxy',
    'refutes_correction_credit_proxy',
    'supports_status_failure_proxy',
]

REFERENCE_FLAG_COLUMNS = ['reference_is_nei', 'reference_is_refutes', 'reference_is_supports']

def existing(cols, df):
    return [c for c in cols if c in df.columns]

def pretty(name):
    return name.replace('_proxy', '').replace('_', ' ').title()

def save_heatmap(table, path, title, vmin=None, vmax=None, cmap='viridis', annotate=True):
    fig, ax = plt.subplots(figsize=(max(7.5, 0.8 * len(table.columns)), max(4.5, 0.45 * len(table.index))))
    image = ax.imshow(table.values, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels([pretty(c) for c in table.columns], rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels(table.index, fontsize=9)
    if annotate:
        for i in range(table.shape[0]):
            for j in range(table.shape[1]):
                val = table.iloc[i, j]
                if pd.notna(val):
                    ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=240)
    plt.show()

def zscore(df):
    return (df - df.mean(axis=0)) / df.std(axis=0, ddof=0).replace(0, np.nan)

def run_pca(values, n_components=4, max_missing_share=0.5):
    numeric = values.apply(pd.to_numeric, errors='coerce')
    missing_share = numeric.isna().mean()
    keep = missing_share[missing_share <= max_missing_share].index.tolist()
    numeric = numeric[keep]
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
    matrix = zscore(numeric.fillna(numeric.mean(axis=0))).dropna(axis=1, how='any')
    if matrix.empty:
        raise ValueError('No usable numeric columns remain for PCA.')
    u, s, vt = np.linalg.svd(matrix.to_numpy(), full_matrices=False)
    k = min(n_components, vt.shape[0])
    names = [f'PC{i+1}' for i in range(k)]
    explained = (s ** 2) / (len(matrix) - 1)
    explained_ratio = explained / explained.sum()
    loadings = pd.DataFrame(vt[:k].T, index=matrix.columns, columns=names)
    variance = pd.DataFrame({'component': names, 'explained_variance_ratio': explained_ratio[:k]})
    scores = pd.DataFrame(u[:, :k] * s[:k], index=matrix.index, columns=names)
    return loadings, variance, scores, matrix.columns.tolist()

## Experiment 0: Data and Output Audit

This experiment checks what data we actually analyzed.

这个实验确认我们实际分析的数据是什么。

The expected response file contains 39,996 rows, corresponding to 19,998 FEVER dev items times two target models.

预期 response file 有 39,996 行，对应 19,998 个 FEVER dev items 乘以两个 target models。

The behavior matrix should contain 239,976 rows, corresponding to 39,996 responses times six evaluator variants.

behavior matrix 应该有 239,976 行，对应 39,996 responses 乘以六个 evaluator variants。

In [ ]:
behavior = pd.read_csv(BEHAVIOR_PATH)
n_responses = sum(1 for _ in RESPONSE_PATH.open()) if RESPONSE_PATH.exists() else None

audit = pd.DataFrame({
    'quantity': [
        'response_rows',
        'behavior_matrix_rows',
        'target_models',
        'evaluator_variants',
        'reference_labels',
    ],
    'value': [
        n_responses,
        len(behavior),
        ', '.join(sorted(behavior['model_name'].dropna().astype(str).unique())),
        ', '.join(sorted(behavior['variant'].dropna().astype(str).unique())),
        ', '.join(sorted(behavior['reference_label'].dropna().astype(str).unique())),
    ]
})
audit.to_csv(OUTDIR / 'table_00_data_audit.csv', index=False)
audit

## Experiment 1: Shared Outcome-Level Comparison

This experiment compares quantities that every evaluator variant produces.

这个实验比较所有 evaluator variants 都会产生的共同输出。

This layer can include baselines because baselines and CALE variants all have final scores and quality tiers.

这一层可以包含 baselines，因为 baselines 和 CALE variants 都有 final scores 和 quality tiers。

However, quality tiers should not be interpreted as factual-status labels.

但是 quality tiers 不能被解释为 factual-status labels。

In [ ]:
shared_summary = (
    behavior.groupby('variant', dropna=False)
    .agg(
        n=('final_score', 'size'),
        mean_final_score=('final_score', 'mean'),
        mean_uncertainty=('uncertainty', 'mean'),
    )
    .sort_values('mean_final_score', ascending=False)
)
shared_summary.round(3).to_csv(OUTDIR / 'table_01_shared_outcome_summary.csv')
shared_summary.round(3)

In [ ]:
quality_dist = (
    behavior.groupby(['variant', 'quality_label'], dropna=False)
    .size()
    .rename('count')
    .reset_index()
)
quality_dist['share'] = quality_dist['count'] / quality_dist.groupby('variant')['count'].transform('sum')
quality_pivot = quality_dist.pivot_table(index='variant', columns='quality_label', values='share', fill_value=0)
quality_pivot.to_csv(OUTDIR / 'table_02_quality_tier_distribution.csv')
quality_pivot.round(3)

## Experiment 2: Diagnostic Availability and Behavior Profiles

This experiment asks which evaluator variants produce construct-level diagnostic variables.

这个实验考察哪些 evaluator variants 产生了 construct-level diagnostic variables。

Baselines can be compared at the shared outcome level, but they do not produce CALE construct subscores.

Baselines 可以在 shared outcome 层面比较，但它们不产生 CALE construct subscores。

This is not only a limitation; it is part of the measurement argument for CALE.

这不只是限制，也是 CALE measurement argument 的一部分。

In [ ]:
behavior_cols = existing(CORE_BEHAVIOR_COLUMNS, behavior)
proxy_cols = existing(PROXY_COLUMNS, behavior)
analysis_cols = behavior_cols + proxy_cols

availability = behavior.groupby('variant')[analysis_cols].apply(lambda g: g.notna().mean())
availability.to_csv(OUTDIR / 'table_03_behavior_variable_availability.csv')
save_heatmap(availability, OUTDIR / 'fig_01_behavior_variable_availability.png', 'Behavior Variable Availability by Evaluator Variant', vmin=0, vmax=1, cmap='Greys')
availability.round(2)

In [ ]:
cale_variants = ['generic_cale', 'attack_aware_cale', 'full_attack_aware_cale']
cale = behavior[behavior['variant'].isin(cale_variants)].copy()

profile = cale.groupby('variant')[analysis_cols].mean().reindex(cale_variants)
profile.to_csv(OUTDIR / 'table_04_cale_behavior_profile_by_variant.csv')
save_heatmap(profile, OUTDIR / 'fig_02_cale_behavior_profile_by_variant.png', 'CALE Behavior Profile by Variant', vmin=0, vmax=1, cmap='viridis')
profile.round(3)

In [ ]:
diff_cols = [
    'claim_status_recognition',
    'correction_accuracy',
    'evidence_grounding',
    'source_faithfulness',
    'supports_status_failure_proxy',
]
diff_cols = existing(diff_cols, profile)
variant_diff = profile[diff_cols].copy()
variant_diff.loc['full_minus_generic'] = profile.loc['full_attack_aware_cale', diff_cols] - profile.loc['generic_cale', diff_cols]
variant_diff.loc['attack_minus_generic'] = profile.loc['attack_aware_cale', diff_cols] - profile.loc['generic_cale', diff_cols]
variant_diff.to_csv(OUTDIR / 'table_05_variant_difference_summary.csv')
variant_diff.round(3)

## Experiment 3: Latent Organization of CALE Behavior

This experiment asks whether CALE behavior variables show coherent low-dimensional structure.

这个实验考察 CALE behavior variables 是否呈现 coherent low-dimensional structure。

We exclude reference-label indicator columns because they describe item condition rather than evaluator behavior.

我们排除 reference-label indicator columns，因为它们描述的是 item condition，而不是 evaluator behavior。

We also focus PCA on CALE variants because baselines do not produce construct-level observables.

我们也把 PCA 聚焦在 CALE variants 上，因为 baselines 不产生 construct-level observables。

In [ ]:
pca_cols = [c for c in analysis_cols if c not in REFERENCE_FLAG_COLUMNS]
pca_input = cale[pca_cols].copy()
loadings, variance, scores, used_cols = run_pca(pca_input, n_components=4, max_missing_share=0.5)

loadings.to_csv(OUTDIR / 'table_06_cale_only_pca_loadings.csv')
variance.to_csv(OUTDIR / 'table_07_cale_only_pca_explained_variance.csv', index=False)
scores.to_csv(OUTDIR / 'table_08_cale_only_pca_scores.csv')

variance['cumulative'] = variance['explained_variance_ratio'].cumsum()
variance.round(4)

In [ ]:
save_heatmap(loadings, OUTDIR / 'fig_03_cale_only_pca_loadings_heatmap.png', 'CALE-Only PCA Loadings', vmin=-1, vmax=1, cmap='RdBu_r')
loadings.round(3)

In [ ]:
top_loading_rows = []
for pc in loadings.columns[:4]:
    ranked = loadings[pc].sort_values(key=lambda s: s.abs(), ascending=False).head(8)
    for variable, loading in ranked.items():
        top_loading_rows.append({'component': pc, 'variable': variable, 'loading': loading, 'abs_loading': abs(loading)})
top_loadings = pd.DataFrame(top_loading_rows)
top_loadings.to_csv(OUTDIR / 'table_09_top_pca_loadings.csv', index=False)
top_loadings.round(3)

In [ ]:
corr = pca_input[used_cols].apply(pd.to_numeric, errors='coerce').corr()
corr.to_csv(OUTDIR / 'table_10_cale_behavior_correlation_matrix.csv')
save_heatmap(corr, OUTDIR / 'fig_04_cale_behavior_correlation_heatmap.png', 'CALE Behavior Correlation Matrix', vmin=-1, vmax=1, cmap='RdBu_r', annotate=False)
corr.round(3)

## Experiment 4: Paper-Facing Result Summary

The current neutral full analysis supports a measurement-centered result story.

当前 neutral full analysis 支持一个以 measurement 为中心的结果叙事。

First, shared outcomes can compare baselines and CALE variants, but shared outcomes alone do not reveal construct-level behavior.

第一，shared outcomes 可以比较 baselines 和 CALE variants，但 shared outcomes 本身不能揭示 construct-level behavior。

Second, CALE variants produce construct-level diagnostic variables that direct baselines do not produce.

第二，CALE variants 产生了 direct baselines 不产生的 construct-level diagnostic variables。

Third, CALE behavior variables show interpretable low-dimensional organization.

第三，CALE behavior variables 呈现出可解释的低维组织结构。

Fourth, variant differences are localized in theoretically meaningful dimensions such as evidence grounding, correction accuracy, and supported-claim boundary behavior.

第四，variant differences 集中在 evidence grounding、correction accuracy 和 supported-claim boundary behavior 等有理论意义的维度上。

In [ ]:
summary_lines = [
    f"Behavior matrix rows: {len(behavior):,}.",
    f"CALE-only PCA PC1-PC4 cumulative variance: {variance['explained_variance_ratio'].sum():.3f}.",
    "PC1 is dominated by source faithfulness, evidence grounding, claim-status recognition, and correction accuracy.",
    "PC2 is dominated by error rejection and framing resistance.",
    "Full attack-aware CALE improves evidence grounding, correction accuracy, claim-status recognition, and supported-claim boundary behavior relative to generic CALE.",
]
summary_text = "\n".join(f"- {line}" for line in summary_lines)
(OUTDIR / 'paper_result_summary.md').write_text(summary_text, encoding='utf-8')
print(summary_text)